In [ ]:
import json
import time
from pathlib import Path

In [ ]:
from rich import print

In [ ]:
from langchain_chroma.vectorstores import cosine_similarity
from sentence_transformers import SentenceTransformer,CrossEncoder

In [ ]:
DATA_PATH = Path("/home/roshan-yadav/Desktop/TheOneEye/Prospect Involved In Post Extractor/intentScoreImprovement/ProspectPostActivity.json")
CACHE_FOLDER = DATA_PATH.parent / "cache"

In [ ]:
QUERY = """
Product Description:
TITAN functions as a centralized Test Lifecycle Management (TLM) platform for the global automotive industry and Testing, Inspection, and Certification (TIC) labs, specifically engineered to satisfy ISO/IEC 17025 competence requirements and FMVSS regulatory mandates. The system addresses fragmented tool landscapes by consolidating test request intake, resource allocation, and automated report generation within a unified digital audit trail. Core solution capabilities include real-time asset readiness tracking, dynamic scheduling of environmental chambers and proving grounds, and automated data post-processing to eliminate manual transcription errors. By enforcing standardized test templates and configurable workflows, the platform mitigates risks associated with tribal knowledge dependency, data integrity breaches, and non-compliance penalties during safety-critical verification and validation phases. TITAN provides governance controls for technical managers and quality assurance officers, ensuring requirement traceability from initial test planning through to final certification. The architecture optimizes operational efficiency for OEMs and Tier-1 suppliers by synchronizing test article oversight with facility management, thereby reducing project lead times and securing business continuity through robust metadata-rich knowledge retention and automated KPI reporting.

Target customer: automotive manufacturers

Interested signals:
- vehicle manufacturing
- automotive supply chain
- EV production
- automotive technology
"""

In [ ]:
embedding_model = SentenceTransformer("./models/bge-large-en-v1.5")
cross_encoder = CrossEncoder("./models/bge-reranker-large")

In [ ]:
with open(DATA_PATH, "r") as file:
    raw_data = json.load(file)

In [ ]:
scored_data = []

query_vec = embedding_model.encode(QUERY)

for index, post in enumerate(raw_data["all_posts"]):
    vec1 = embedding_model.encode(post["post_content"])
    similarity = cosine_similarity([query_vec], [vec1])[0,0]
    post["similarity"] = similarity
    scored_data.append(post)

scored_data.sort(key=lambda x: x["similarity"], reverse=True)
print(scored_data)

In [ ]:

top_k_data = scored_data[:5]
cross_encoder_input = []

for index, post in enumerate(top_k_data):
    cross_encoder_input.append([QUERY, post["post_content"]])

scores = cross_encoder.predict(cross_encoder_input)
print(scores)

cross_encoder_result = []

for cross_encoder_score, post in zip(scores, top_k_data):
    post["cross_encoder_score"] = cross_encoder_score
    cross_encoder_result.append(post)

print(cross_encoder_result)

In [ ]:
data2 = """
i am in automotive industry and i am interested in the product which is related to automotive industry.
"""

cross_encoder_input = [QUERY, data2]
cross_encoder_score = cross_encoder.predict([cross_encoder_input])
print(cross_encoder_score)